# Phase 4C.2H.1 — official development runner

Self-staging launcher for the exact sealed code snapshot. Default mode is the two-fit pilot: one fit per recipe, both at outer fold 0 / inner fold 0 / seed 42. A valid completed pilot receipt is reused by the official 120-inner + 30-outer run. This notebook never reads the retired Phase 4C.2G locked test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

MODE = 'pilot'  # 'preflight', 'pilot', or explicit 'all'
ALLOW_FULL_150_FITS = False
DEVICE = 'cuda'
SOURCE_COMMIT = '75568d1c03d89e02ad654df75c969924b857ee78'
CODE_ARCHIVE_NAME = 'phase_4c2h_code_75568d1.tar.gz'
CODE_ARCHIVE_BYTES = 9509956
CODE_ARCHIVE_SHA256 = '2bd393873be3291084f86a329a34566bd9aa46c0efd8aa9f155e47571b132c18'
DATASET_ARCHIVE_NAME = 'phase_4c1_binary_n250_reusable.tar'
DATASET_ARCHIVE_BYTES = 724633600
DATASET_ARCHIVE_SHA256 = 'd49a106f0c4991ca8d79776277cbf7331df209157725c438288720dc42226a27'
DATASET_MANIFEST_SHA256 = '411e35da80843a10f7bb22e33efac8a4d1db70110dbda91cadbbf39532c9312d'
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/forensics-web-lab')
CODE_ARCHIVE = DRIVE_ROOT / 'phase_4c2h/inputs' / CODE_ARCHIVE_NAME
DATASET_ARCHIVE = DRIVE_ROOT / 'phase_4c1/inputs' / DATASET_ARCHIVE_NAME
STAGE = Path('/content/phase4c2h-code-75568d1')
BUNDLE = Path('/content/phase4c2h-development-bundle')
OUTPUT = DRIVE_ROOT / 'phase_4c2h/runs/execution_75568d1'


In [ ]:
import hashlib, importlib.metadata, json, os, platform, shutil, tarfile
from pathlib import PurePosixPath
from packaging.version import Version

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): digest.update(chunk)
    return digest.hexdigest()

def verify_file(path, expected_bytes, expected_sha256):
    assert path.is_file() and not path.is_symlink(), f'Missing regular file: {path}'
    assert path.stat().st_size == expected_bytes, f'Byte mismatch: {path}'
    assert sha256_file(path) == expected_sha256, f'SHA-256 mismatch: {path}'

def safe_extract(archive_path, destination, mode):
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True)
    with tarfile.open(archive_path, mode) as archive:
        members = archive.getmembers()
        for member in members:
            posix = PurePosixPath(member.name)
            assert member.isfile(), f'Non-regular archive member: {member.name}'
            assert not posix.is_absolute() and '..' not in posix.parts and '\\' not in member.name
        archive.extractall(destination, members=members)

assert MODE in {'preflight', 'pilot', 'all'}
assert MODE != 'all' or ALLOW_FULL_150_FITS, 'Full 150-fit run requires explicit opt-in'
verify_file(CODE_ARCHIVE, CODE_ARCHIVE_BYTES, CODE_ARCHIVE_SHA256)
verify_file(DATASET_ARCHIVE, DATASET_ARCHIVE_BYTES, DATASET_ARCHIVE_SHA256)
assert Version(platform.python_version()) >= Version('3.10')
minimums = {'torch':'2.2.0','torchvision':'0.17.0','numpy':'1.26.0','PyYAML':'6.0','Pillow':'10.2.0','scikit-learn':'1.4.0'}
for package, minimum in minimums.items():
    actual = importlib.metadata.version(package)
    assert Version(actual) >= Version(minimum), f'{package} {actual} < {minimum}'
import torch
assert DEVICE != 'cuda' or torch.cuda.is_available(), 'Select a Colab GPU runtime'
assert shutil.disk_usage('/content').free >= 5_000_000_000, 'At least 5 GB free local disk required'
safe_extract(CODE_ARCHIVE, STAGE, 'r:gz')
safe_extract(DATASET_ARCHIVE, BUNDLE, 'r:')
verify_file(BUNDLE / 'manifest_pilot_a_option_p.csv', 126503, DATASET_MANIFEST_SHA256)
snapshot = json.loads((STAGE / 'SNAPSHOT_MANIFEST.json').read_text())
assert snapshot['source_commit'] == SOURCE_COMMIT and len(snapshot['members']) == 11
OUTPUT.mkdir(parents=True, exist_ok=True)
print({'mode': MODE, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'output': str(OUTPUT)})


In [ ]:
import subprocess, sys
command = [
    sys.executable, '-m', 'ml.training.run_phase_4c2h',
    '--mode', MODE,
    '--protocol', str(STAGE / 'ml/configs/phase_4c2h_development_nested_cv.yaml'),
    '--bundle', str(BUNDLE),
    '--dataset-archive', str(DATASET_ARCHIVE),
    '--weights-path', str(STAGE / 'models/research/pretrained/mobilenet_v3_small-047dcff4.pth'),
    '--output', str(OUTPUT),
    '--snapshot-manifest', str(STAGE / 'SNAPSHOT_MANIFEST.json'),
    '--code-snapshot-archive', str(CODE_ARCHIVE),
    '--code-snapshot-sha256', CODE_ARCHIVE_SHA256,
    '--code-snapshot-bytes', str(CODE_ARCHIVE_BYTES),
    '--source-commit', SOURCE_COMMIT, '--device', DEVICE,
]
subprocess.run(command, cwd=STAGE, check=True)


In [ ]:
progress = json.loads((OUTPUT / 'progress.json').read_text())
pilot_receipts = sorted((OUTPUT / 'fits/inner').glob('*/fit_receipt.json'))
print(json.dumps(progress, indent=2, sort_keys=True))
print('Completed pilot/inner receipts:', len(pilot_receipts))
for path in pilot_receipts:
    receipt = json.loads(path.read_text())
    print(path.parent.name, {'epochs': receipt['epochs_completed'], 'wall_seconds': receipt['wall_seconds'], 'device': receipt['device']})
print('Resume: rerun this notebook with the same OUTPUT and exact inputs; valid completed receipts are hash-verified and skipped.')
